# LandslideGuard - Stage 2 Kaggle GPU Environment Setup

This notebook prepares and **verifies** the Kaggle environment for Stage 2 (U-Net
training). It **does not train** the model. Once every check in Section 11
passes, open a separate Stage-2 training notebook.

**What this notebook does:**

1. Prints environment + GPU information.
2. Clones the LandslideGuard GitHub repo (or attaches an already-uploaded copy).
3. Adds the repo to `sys.path` and imports the frozen Stage-1 modules.
4. Verifies dependencies (installs h5py only if missing - never touches Kaggle's torch).
5. Locates the Landslide4Sense dataset supplied to Kaggle separately.
6. Loads the Stage-1 train-only normalization statistics (never recomputes them).
7. Builds `Landslide4SenseDataset` and `DataLoader` for train/valid/test.
8. Pulls one real batch per split and moves it to CUDA.
9. Reports PASS/FAIL for every critical check.

See [docs/kaggle_setup.md](../docs/kaggle_setup.md) for step-by-step instructions.


## SECTION 01 - Environment Information

Prints the Python + package versions and the working directory so this run is
reproducible.


In [ ]:
import os, sys, platform, importlib
print("Python           :", sys.version.split()[0])
print("Platform         :", platform.platform())
print("Working directory:", os.getcwd())

for mod in ["torch", "torchvision", "h5py", "numpy", "pandas",
            "matplotlib", "yaml"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:12s} {getattr(m, '__version__', 'unknown')}")
    except Exception as e:
        print(f"{mod:12s} MISSING ({type(e).__name__})")


## SECTION 02 - GPU Verification

Kaggle's accelerator setting decides whether `torch.cuda.is_available()` is
true. If it is false, switch **Session options -> Accelerator** to a GPU
(e.g. T4 x2) and re-run all cells before continuing. This notebook does not
start training; it only verifies that a GPU exists.


In [ ]:
import torch
gpu_ok = torch.cuda.is_available()
print("DEVICE")
print("------")
print("cuda" if gpu_ok else "cpu (no GPU attached)")
print()
print("GPU")
print("---")
if gpu_ok:
    n = torch.cuda.device_count()
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / 1024**3
        print(f"  [{i}] {props.name}  ({mem_gb:.1f} GiB)")
    print(f"  device count : {n}")
    print(f"  cuda version : {torch.version.cuda}")
else:
    print("  WARNING: no GPU available. Enable the GPU accelerator "
          "and re-run before Stage 2.")
DEVICE = torch.device("cuda" if gpu_ok else "cpu")
print()
print("torch build      :", torch.__version__)
print("cudnn enabled    :", torch.backends.cudnn.is_available())


## SECTION 03 - Clone GitHub Repository

**Set `GITHUB_REPO` to your repository URL** (the exact one used by
`git clone`). The notebook clones it into `/kaggle/working/LandslideGuard` and
records the resolved commit hash + branch, making every Kaggle run
traceable to a specific project version.

If you cannot use `git clone` from Kaggle for some reason, set
`GITHUB_REPO = None` and provide `LOCAL_REPO_PATH` (e.g., a code-only Kaggle
dataset).


In [ ]:
# ==== CONFIGURATION - set these two variables ====
GITHUB_REPO = "https://github.com/Aryan2080/Landslide-Guard"  # e.g. "https://github.com/<owner>/<repo>.git"
GITHUB_REF  = "main"        # branch, tag, or commit hash to check out
LOCAL_REPO_PATH = None      # optional fallback if you uploaded code as a Kaggle dataset
# =================================================

import os, subprocess
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK / "LandslideGuard"

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"command failed with exit {r.returncode}: {' '.join(cmd)}")

if GITHUB_REPO:
    if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
        run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
        run(["git", "checkout", GITHUB_REF],       cwd=REPO_DIR)
        run(["git", "pull", "--ff-only"],          cwd=REPO_DIR)
    else:
        if REPO_DIR.exists():
            import shutil; shutil.rmtree(REPO_DIR)
        run(["git", "clone", "--depth=1", "--branch", GITHUB_REF,
             GITHUB_REPO, str(REPO_DIR)])
elif LOCAL_REPO_PATH:
    REPO_DIR = Path(LOCAL_REPO_PATH)
    assert REPO_DIR.is_dir(), f"LOCAL_REPO_PATH does not exist: {REPO_DIR}"
else:
    raise RuntimeError("Set either GITHUB_REPO or LOCAL_REPO_PATH.")

print()
print("repository path :", REPO_DIR)
if (REPO_DIR / ".git").exists():
    head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                          capture_output=True, text=True).stdout.strip()
    branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                            cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
    print("git commit      :", head)
    print("git branch      :", branch)


## SECTION 04 - Python Path Configuration

Puts the cloned repo on `sys.path` and imports the frozen Stage-1 modules.
The modules themselves are **not** modified for Kaggle - they are the same
files that were verified locally.


In [ ]:
import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.detection import preprocessing, dataset
from src.detection.preprocessing import NormalizationStats, preprocess_pair
from src.detection.dataset       import (Landslide4SenseDataset,
                                         build_all_splits, build_dataloader)

print("preprocessing module:", preprocessing.__file__)
print("dataset       module:", dataset.__file__)


## SECTION 05 - Dependency Verification

Checks each dependency and pip-installs **only what is genuinely missing**.
We never reinstall `torch` / `torchvision`: Kaggle's images ship a CUDA-
enabled PyTorch and reinstalling from PyPI would swap in a CPU wheel.


In [ ]:
import importlib, subprocess, sys
REQUIRED = ["h5py", "yaml", "matplotlib", "numpy", "pandas"]
missing = []
for m in REQUIRED:
    try:
        importlib.import_module(m)
    except ImportError:
        missing.append(m)
print("missing (before install):", missing)

PIP_NAMES = {"yaml": "pyyaml"}
if missing:
    pkgs = [PIP_NAMES.get(m, m) for m in missing]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# re-check
still_missing = []
for m in REQUIRED:
    try:
        importlib.import_module(m)
    except ImportError:
        still_missing.append(m)
print("missing (after install) :", still_missing)
assert not still_missing, still_missing


## SECTION 06 - Dataset Location Verification

**Set `DATA_ROOT` to the directory that directly contains `TrainData/`,
`ValidData/`, and `TestData/`.** The cell below auto-detects the two common
layouts:

1. `<DATA_ROOT>/TrainData/img/*.h5`   (flat)
2. `<DATA_ROOT>/TrainData/TrainData/img/*.h5`   (archive-nested)

and picks whichever it finds. Expected counts: train 3799 / valid 245 /
test 800. Anything else is reported but does not silently pass.


In [ ]:
from pathlib import Path

# ==== CONFIGURATION ====
DATA_ROOT = "/kaggle/input/landslide4sense"   # <-- change to the exact path Kaggle mounted
# =======================

root = Path(DATA_ROOT)
assert root.is_dir(), f"DATA_ROOT does not exist: {root}"

def resolve_split_dir(root, split):
    """Return (img_dir, mask_dir) for a split, handling flat + nested layouts."""
    candidates = [
        (root / split / "img",             root / split / "mask"),
        (root / split / split / "img",     root / split / split / "mask"),
    ]
    for img_d, mask_d in candidates:
        if img_d.is_dir() and mask_d.is_dir():
            return img_d, mask_d
    raise FileNotFoundError(
        f"Could not find img/ + mask/ under {root/split}. Checked: {candidates}"
    )

SPLIT_PATHS = {name: resolve_split_dir(root, name)
               for name in ["TrainData", "ValidData", "TestData"]}

EXPECTED = {"TrainData": 3799, "ValidData": 245, "TestData": 800}

print(f"{'split':10s} {'img files':>10s} {'mask files':>11s}  path")
counts = {}
for name, (img_d, mask_d) in SPLIT_PATHS.items():
    n_img  = sum(1 for p in img_d.iterdir() if p.suffix == ".h5")
    n_mask = sum(1 for p in mask_d.iterdir() if p.suffix == ".h5")
    counts[name] = (n_img, n_mask)
    mark = "OK" if n_img == n_mask == EXPECTED[name] else "!!"
    print(f"{name:10s} {n_img:10d} {n_mask:11d}  {img_d}  [{mark}]")


## SECTION 07 - Stage-1 Source Verification

Confirms Kaggle is running the **same** `preprocessing.py` and `dataset.py`
that were verified in Stage 1 by comparing SHA-256 hashes with the copy on
disk in the cloned repo.


In [ ]:
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

files = {
    "preprocessing.py": REPO_DIR / "src" / "detection" / "preprocessing.py",
    "dataset.py"      : REPO_DIR / "src" / "detection" / "dataset.py",
}
for name, path in files.items():
    assert path.exists(), f"missing {path}"
    print(f"{name:20s} {sha256(path)}  {path}")


## SECTION 08 - Normalization Statistics Verification

Loads the frozen train-only normalization statistics from the cloned repo.
**These are never recomputed on Kaggle** - that would leak validation/test
information and violate the Stage-1 contract.


In [ ]:
import json
norm_path = REPO_DIR / "outputs" / "detection" / "data_verification" / "normalization_statistics.json"
assert norm_path.is_file(), f"Stage-1 normalization stats missing: {norm_path}"

norm = json.loads(norm_path.read_text())
assert norm.get("method") == "per_channel_zscore"
assert norm.get("computed_on") == "train_split_only"
assert len(norm["mean"]) == 14 and len(norm["std"]) == 14

stats = NormalizationStats.from_json(norm_path)
print(f"{'ch':>2}  {'mean':>10}  {'std':>10}")
for c in range(14):
    print(f"{c:2d}  {stats.mean[c]:10.4f}  {stats.std[c]:10.4f}")
print()
print("computed_on   :", norm["computed_on"])
print("n_train_files :", norm.get("n_train_files"))


## SECTION 09 - Dataset / DataLoader

Instantiates `Landslide4SenseDataset` for each split using the resolved paths.
No custom logic - imports the exact Stage-1 implementation from the repo.


In [ ]:
from src.detection.dataset import Landslide4SenseDataset

datasets = {}
for split_name, (img_d, mask_d) in SPLIT_PATHS.items():
    ds_split = {"TrainData": "train", "ValidData": "valid",
                "TestData": "test"}[split_name]
    datasets[ds_split] = Landslide4SenseDataset(
        img_dir=img_d, mask_dir=mask_d, stats=stats,
        split=ds_split, augment_seed=42)

for name, d in datasets.items():
    r = d.report
    print(f"{name:6s} paired={r.n_paired:4d}  augment={d._aug is not None}")


## SECTION 10 - Real Batch Verification

Pulls one batch from each `DataLoader`, checks shape/dtype/finiteness/mask
values, and moves the batch onto the GPU (if one is attached). Nothing is
trained.


In [ ]:
from src.detection.dataset import build_dataloader

B = 8
batch_info = {}
for name, d in datasets.items():
    loader = build_dataloader(d, batch_size=B, num_workers=0)
    xb, yb = next(iter(loader))
    assert xb.shape == (B, 14, 128, 128), xb.shape
    assert yb.shape == (B, 128, 128), yb.shape
    assert xb.dtype == torch.float32 and yb.dtype == torch.float32
    assert torch.isfinite(xb).all()
    unique_mask = sorted(set(torch.unique(yb).tolist()))
    assert set(unique_mask).issubset({0.0, 1.0}), unique_mask
    xb_dev = xb.to(DEVICE, non_blocking=True)
    yb_dev = yb.to(DEVICE, non_blocking=True)
    batch_info[name] = {
        "shape_x": tuple(xb.shape),
        "shape_y": tuple(yb.shape),
        "dtype_x": str(xb.dtype),
        "dtype_y": str(yb.dtype),
        "on_device": str(xb_dev.device),
        "mask_unique": unique_mask,
    }
    print(f"{name:6s} x={tuple(xb.shape)} y={tuple(yb.shape)} "
          f"device={xb_dev.device} finite=True mask_unique={unique_mask}")


## SECTION 11 - Final Kaggle Environment Validation

PASS/FAIL summary. Only proceed to Stage 2 if the last line reads
**`KAGGLE ENVIRONMENT READY`**.


In [ ]:
checks = []

def rec(name, ok, detail=""):
    checks.append((name, ok, detail))

rec("Python (>=3.10)", sys.version_info >= (3, 10),
    f"{sys.version.split()[0]}")
rec("PyTorch",         hasattr(torch, "__version__"),
    torch.__version__)
rec("GPU (CUDA)",      torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")
rec("GitHub clone",    (REPO_DIR / ".git").exists() or (REPO_DIR / "src" / "detection").is_dir(),
    str(REPO_DIR))
rec("Source imports",  callable(getattr(preprocessing, "preprocess_pair", None))
    and callable(getattr(dataset, "build_dataloader", None)), "")
rec("Dataset found",   all((p[0].is_dir() and p[1].is_dir()) for p in SPLIT_PATHS.values()),
    str(root))
rec("Train split",     counts["TrainData"] == (EXPECTED["TrainData"], EXPECTED["TrainData"]),
    f"{counts['TrainData']}")
rec("Validation split",counts["ValidData"] == (EXPECTED["ValidData"], EXPECTED["ValidData"]),
    f"{counts['ValidData']}")
rec("Test split",      counts["TestData"] == (EXPECTED["TestData"], EXPECTED["TestData"]),
    f"{counts['TestData']}")
rec("14 channels",     batch_info["train"]["shape_x"][1] == 14, "")
rec("Normalization",   norm.get("computed_on") == "train_split_only"
    and len(norm["mean"]) == 14, "")
rec("DataLoader",      all(v["shape_x"] == (8, 14, 128, 128) for v in batch_info.values()), "")
rec("Real batch",      all(v["mask_unique"] and set(v["mask_unique"]).issubset({0.0, 1.0})
                          for v in batch_info.values()), "")
rec("CUDA tensor transfer",
    all(v["on_device"].startswith("cuda") for v in batch_info.values())
    if torch.cuda.is_available() else True,
    "skipped (no GPU)" if not torch.cuda.is_available() else "")

print("=" * 50)
print("KAGGLE STAGE-2 ENVIRONMENT VALIDATION")
print("=" * 50)
width = max(len(n) for n, _, _ in checks)
for name, ok, detail in checks:
    dots = "." * max(2, 32 - len(name))
    tag = "PASS" if ok else "FAIL"
    print(f"{name:{width}s} {dots} {tag}  {detail}")

fails = [n for n, ok, _ in checks if not ok]
print()
if fails:
    print("BLOCKED - failed checks:", fails)
    print("Fix the failing items and re-run all cells before Stage 2.")
else:
    print("KAGGLE ENVIRONMENT READY")
    print("Proceed to Stage 2: create a separate training notebook (e.g. "
          "notebooks/03_detection_training_kaggle.ipynb).")
